# Lab 1: W&B tracking with a Random Forest on the Wine dataset

In this version, I use the Wine dataset from scikit-learn and train a Random Forest classifier. This makes my lab different from the original notebook, which used XGBoost with the Dermatology dataset.

In [1]:
import numpy as np
import pandas as pd
import wandb

from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/vigneshraja/.netrc.
wandb: Currently logged in as: vigneshvrs5 (vigneshvrs5-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
config = {
    "test_size": 0.2,
    "random_state": 42,
    "n_estimators": 180,
    "max_depth": 8,
    "min_samples_split": 4,
}

run = wandb.init(
    project="Lab1-wine-random-forest",
    name="wine_random_forest_baseline",
    config=config,
)

wine = load_wine(as_frame=True)
X = wine.data.copy()
y = wine.target.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=wandb.config.test_size,
    random_state=wandb.config.random_state,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(
    n_estimators=wandb.config.n_estimators,
    max_depth=wandb.config.max_depth,
    min_samples_split=wandb.config.min_samples_split,
    random_state=wandb.config.random_state,
)
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)
report = classification_report(y_test, predictions, target_names=wine.target_names)

wandb.log({"test_accuracy": accuracy})
run.summary["test_accuracy"] = accuracy
run.summary["dataset"] = "scikit-learn wine"
run.summary["model"] = "RandomForestClassifier"

wandb.sklearn.plot_confusion_matrix(y_test, predictions, labels=wine.target_names)

importance_df = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": model.feature_importances_,
    }
).sort_values("importance", ascending=False)

wandb.log(
    {
        "feature_importance": wandb.Table(dataframe=importance_df),
        "top_feature_importance": wandb.plot.bar(
            wandb.Table(dataframe=importance_df.head(8)),
            "feature",
            "importance",
            title="Top 8 feature importances",
        ),
    }
)

print(f"Test accuracy: {accuracy:.4f}")
print(report)

run.finish()

Test accuracy: 1.0000
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



test_accuracy,▁
dataset,scikit-learn wine
model,RandomForestClassifi...
test_accuracy,1
